# Typecast Anime Voice Test

Test Typecast TTS for anime-style voice generation.

## What this tests
- Typecast API for expressive anime voices
- SmartPrompt emotion control
- Real-time audio playback with sounddevice

## Voice used
- **Model**: ssfm-v21 (Typecast's latest)
- **Voice ID**: tc_6359e7f6467f9e240b68292c
- **Prompt**: SmartPrompt with emotion control

## Why not in production
Typecast was evaluated but ElevenLabs provided better voice quality and lower latency. This code remains as reference for anime voice synthesis.

## Dependencies
```bash
pip install typecast-python sounddevice numpy
```

## Environment
Requires `TYPECAST_API_KEY` in `.env`

In [ ]:
import os
from dotenv import load_dotenv
import io
import wave
import numpy as np
import sounddevice as sd
from typecast import Typecast
from typecast.models import TTSRequest, SmartPrompt

# Load API key from environment
load_dotenv()

# Initialize Typecast client
api_key = os.getenv("TYPECAST_API_KEY")
if not api_key:
    print("Warning: TYPECAST_API_KEY not found in .env")
    
client = Typecast(api_key=api_key)
print("✓ Typecast client initialized")

In [ ]:
def play_anime_voice(text):
    """
    Synthesize anime voice using Typecast and play immediately.
    
    Args:
        text: Text to synthesize (should include emotion markers)
    """
    print(f"Synthesizing: {text[:50]}...")
    
    try:
        # Call Typecast TTS API
        response = client.text_to_speech(TTSRequest(
            text=text,
            model="ssfm-v21",  # Voice synthesis model
            voice_id="tc_6359e7f6467f9e240b68292c",  # Anime character voice
            prompt=SmartPrompt(emotion_type="smart")  # Auto emotion detection
        ))
        
        # Parse WAV data from response
        audio_data = io.BytesIO(response.audio_data)
        with wave.open(audio_data, 'rb') as wf:
            channels = wf.getnchannels()
            sample_width = wf.getsampwidth()
            framerate = wf.getframerate()
            
            # Determine numpy dtype from sample width
            dtype_map = {1: np.uint8, 2: np.int16, 4: np.int32}
            dtype = dtype_map.get(sample_width, np.int16)
            
            frames = wf.readframes(wf.getnframes())
            audio_array = np.frombuffer(frames, dtype=dtype)
            
            if channels > 1:
                audio_array = audio_array.reshape(-1, channels)
        
        # Play audio
        sd.play(audio_array, samplerate=framerate)
        sd.wait()
        print("Done!")
        
    except Exception as e:
        print(f"Error: {e}")


# Test with tsundere-style dialogue
test_text = (
    "W-what are you staring at?! It's not like I dressed up just for you. "
    "I just grabbed whatever was in my closet! "
    "But... since you're here... I guess you can stay."
)

play_anime_voice(test_text)

In [ ]:
demo_text = (
    "W-what are you staring at?! It's not like I dressed up just for you or anything. "
    "I just grabbed whatever was in my closet! "
    "But... since you're here... I guess you can stay. Just don't get any weird ideas, got it?!"
)

play_anime_voice(demo_text)
